# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [37]:
# imports
from openai import OpenAI
import os
import json
import re
from IPython.display import display, update_display, Markdown

In [19]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
MODEL_CLAUDE = 'claude-sonnet-4-6'
api_key = os.getenv('CLAUDE_API_KEY')
if not api_key or not api_key.startswith('sk-ant-api03-'):
    print("There might be a problem with your OpenAI API key? Please visit the troubleshooting notebook!")
else:
    print("OpenAI API key looks good so far")


OpenAI API key looks good so far


In [20]:
# set up environment
BASE_URL = 'https://api.anthropic.com/v1'
openai = OpenAI(base_url=BASE_URL, api_key=api_key)

In [21]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:



stream =openai.chat.completions.create(
    model=MODEL_CLAUDE,
    messages=[
        {"role": "user", "content": question}
    ],
    stream=True)

response = ""
display_handle = display(Markdown(response), display_id=True)


for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)  


## Code Explanation

### What it does
This single line **yields unique author names** from a collection of books, skipping any books that have no author.

---

### Breaking it Down

```python
yield from {book.get("author") for book in books if book.get("author")}
```

| Part | What it does |
|------|-------------|
| `book.get("author")` | Safely retrieves the `"author"` key, returning `None` if missing (instead of raising a `KeyError`) |
| `for book in books` | Iterates over each book in the collection |
| `if book.get("author")` | Filters out books where author is `None`, empty string, or any falsy value |
| `{ ... }` | A **set comprehension** - automatically removes duplicate authors |
| `yield from` | Yields each item from the set one at a time (used inside a generator function) |

---

### Example

```python
books = [
    {"title": "Dune",       "author": "Frank Herbert"},
    {"title": "Foundation", "author": "Isaac Asimov"},
    {"title": "Dune Messiah", "author": "Frank Herbert"},  # duplicate
    {"title": "Unknown"},                                  # no author
]

# Yields (in no guaranteed order):
# "Frank Herbert"
# "Isaac Asimov"
```

---

### Key Behaviours to Be Aware Of

> **Order is not guaranteed** — because sets are unordered, the authors will be yielded in an unpredictable sequence each time. If order matters, use a list comprehension `[...]` instead.

This pattern is commonly used in **generator functions** to lazily produce a deduplicated stream of values from a collection.

In [23]:


stream =openai.chat.completions.create(
    model=MODEL_CLAUDE,
    messages=[
        {"role": "user", "content": question},
        {"role": "assistant", "content": response},
        {"role": "user", "content": "What is use of the yeild here?"}
    ],
    stream=True)

new_response = ""
display_handle = display(Markdown(new_response), display_id=True)


for chunk in stream:
    new_response += chunk.choices[0].delta.content or ''
    update_display(Markdown(new_response), display_id=display_handle.display_id)

## Purpose of `yield` Here

The `yield` keyword means this code **must live inside a generator function**, and its purpose is to **produce values one at a time** instead of returning them all at once.

---

### `yield from` vs `return`

```python
# With yield from - generator function
def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# With return - regular function
def get_authors(books):
    return {book.get("author") for book in books if book.get("author")}
```

| | `yield from` | `return` |
|---|---|---|
| Returns | A **generator object** | The **whole set** at once |
| Memory | Loads one item at a time | Loads everything into memory |
| Caller gets | An **iterable** they can loop over | A **set** they can use directly |

---

### Honestly, In This Specific Case...

The benefit is **minimal** because the set comprehension `{ ... }` already builds the **entire set in memory** before `yield from` even runs.

So the real memory saving of a generator is already lost at that point.

```python
# If memory efficiency was the true goal, you'd avoid the set
# But then you'd lose deduplication
yield from (book.get("author") for book in books if book.get("author"))
#           ^ parentheses = generator expression, truly lazy, but has duplicates
```

---

### Summary
The `yield` here makes the **function itself** behave as a generator, but the set comprehension inside it **undermines the laziness** benefit. It mainly serves to make the function's output **iterable in a consistent way** with other generators in the codebase.

In [51]:
BASE_OLLAMA_URL = 'http://localhost:11434/v1'

MODEL_OLLAMA = 'gpt-oss:latest'

SYSTEM_PROMPT = """You are an invoice extraction assistant. Your task is to extract structured invoice data from unstructured document text and return a single JSON object. Do not return any text before or after the JSON.

## Input format: coordinate markers

The input text contains inline markers that indicate where each span of text appears in the original document. The pattern is: <@CO_ID_<number>> (e.g. <@CO_ID_1>, <@CO_ID_42>).

- When you extract a value: strip these markers from the text so the stored value is plain text/numbers only (e.g. "ABC LTD", not "ABC<@CO_ID_1> LTD<@CO_ID_2>").
- In "coordinatesUsed": record only the coordinate IDs that contributed to that value, as strings without angle brackets (e.g. ["CO_ID_1", "CO_ID_2"]).

Example: input "ABC<@CO_ID_1> LTD<@CO_ID_2>" → supplierName value is "ABC LTD", and coordinatesUsed.supplierName is ["CO_ID_1", "CO_ID_2"].

## Output: exact JSON structure

Return exactly one JSON object with the following keys. Use the types and formats below. For array fields (taxInfo, merchItemInfo), in coordinatesUsed use a map from index (string "0", "1", ...) to the coordinate object for that element. If a field is not present in the document, omit it or use null.

{
  "supplierName": "string",
  "invoiceId": "string",
  "invoiceDate": "string (dd-mm-yyyy, e.g. 25-12-2024)",
  "totalNetAmount": number,
  "totalTaxAmount": number,
  "totalGrossAmount": number,
  "taxInfo": [
    { "taxType": "string", "taxRate": number, "taxAmount": number }
  ],
  "merchItemInfo": [
    { "itemNumber": "string", "itemDescription": "string", "Quantity": number, "totalAmount": number }
  ],
  "coordinatesUsed": {
    "supplierName": ["CO_ID_xx", "CO_ID_xy",...],
    "invoiceId": ["..."],
    "invoiceDate": ["..."],
    "totalNetAmount": ["..."],
    "totalTaxAmount": ["..."],
    "totalGrossAmount": ["..."],
    "taxInfo": {
      "0": { "taxType": ["..."], "taxRate": ["..."], "taxAmount": ["..."] },
      "1": { "taxType": ["..."], "taxRate": ["..."], "taxAmount": ["..."] }
    },
    "merchItemInfo": {
      "0": { "itemNumber": ["..."], "itemDescription": ["..."], "Quantity": ["..."], "totalAmount": ["..."] },
      "1": { "itemNumber": ["..."], "itemDescription": ["..."], "Quantity": ["..."], "totalAmount": ["..."] }
    }
  }
}
"""

# Sample invoice document with coordinate markers (replace with your own input)
INVOICE_CONTENT = """
                                                      MUKESH<@CO_ID_1> MEDICALS<@CO_ID_2>

Invoice<@CO_ID_3> No<@CO_ID_4> :<@CO_ID_5> 1234-SD<@CO_ID_6>                                        Invoice<@CO_ID_7> Date<@CO_ID_8>:<@CO_ID_9> 12-Mar-26<@CO_ID_10>



Article<@CO_ID_11>          Qty<@CO_ID_12>         Amount<@CO_ID_13>

Dolo<@CO_ID_14> 150<@CO_ID_15>          10<@CO_ID_16>           15<@CO_ID_17>
Vicks<@CO_ID_18>              5<@CO_ID_19>           20<@CO_ID_20>

					Total<@CO_ID_21>      35<@CO_ID_22>
					Vat<@CO_ID_23>         0<@CO_ID_24>
					                35<@CO_ID_25>
"""

BASE_GOOGLE_URL = 'https://generativelanguage.googleapis.com/v1beta'
google_api_key = os.getenv('GOOGLE_API_KEY')
#openai = OpenAI(base_url=BASE_OLLAMA_URL)
#openai = OpenAI(base_url=BASE_URL, api_key=api_key)
openai = OpenAI()
#MODEL='gpt-5.4'
#MODEL='gpt-5-mini'
#MODEL='gpt-5-nano'
MODEL='gpt-4.1'
#openai = OpenAI(base_url=BASE_GOOGLE_URL, api_key=google_api_key)
#MODEL = 'gemini-3.1-flash-lite-preview'

stream = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": INVOICE_CONTENT}
    ],
    stream=True)

ollama_response = ""
display_handle = display(Markdown(ollama_response), display_id=True)


for chunk in stream:
    ollama_response += chunk.choices[0].delta.content or ''
    update_display(Markdown(ollama_response), display_id=display_handle.display_id)

# Pretty-print JSON output
raw = ollama_response.strip()
# Strip markdown code fence if present (e.g. ```json ... ```)
json_match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', raw)
json_str = json_match.group(1).strip() if json_match else raw
try:
    data = json.loads(json_str)
    pretty_json = json.dumps(data, indent=2, ensure_ascii=False)
    update_display(Markdown(f"```json\n{pretty_json}\n```"), display_id=display_handle.display_id)
except json.JSONDecodeError:
    pass  # keep original response if not valid JSON



```json
{
  "supplierName": "MUKESH MEDICALS",
  "invoiceId": "1234-SD",
  "invoiceDate": "12-03-2026",
  "totalNetAmount": 35,
  "totalTaxAmount": 0,
  "totalGrossAmount": 35,
  "taxInfo": [
    {
      "taxType": "Vat",
      "taxRate": 0,
      "taxAmount": 0
    }
  ],
  "merchItemInfo": [
    {
      "itemNumber": "1",
      "itemDescription": "Dolo 150",
      "Quantity": 10,
      "totalAmount": 15
    },
    {
      "itemNumber": "2",
      "itemDescription": "Vicks",
      "Quantity": 5,
      "totalAmount": 20
    }
  ],
  "coordinatesUsed": {
    "supplierName": [
      "CO_ID_1",
      "CO_ID_2"
    ],
    "invoiceId": [
      "CO_ID_6"
    ],
    "invoiceDate": [
      "CO_ID_10"
    ],
    "totalNetAmount": [
      "CO_ID_22"
    ],
    "totalTaxAmount": [
      "CO_ID_24"
    ],
    "totalGrossAmount": [
      "CO_ID_25"
    ],
    "taxInfo": {
      "0": {
        "taxType": [
          "CO_ID_23"
        ],
        "taxRate": [
          "CO_ID_24"
        ],
        "taxAmount": [
          "CO_ID_24"
        ]
      }
    },
    "merchItemInfo": {
      "0": {
        "itemNumber": [],
        "itemDescription": [
          "CO_ID_14",
          "CO_ID_15"
        ],
        "Quantity": [
          "CO_ID_16"
        ],
        "totalAmount": [
          "CO_ID_17"
        ]
      },
      "1": {
        "itemNumber": [],
        "itemDescription": [
          "CO_ID_18"
        ],
        "Quantity": [
          "CO_ID_19"
        ],
        "totalAmount": [
          "CO_ID_20"
        ]
      }
    }
  }
}
```